# Manual Run - ETL iSeller Mitra Fulfillment

> Notebook untuk menjalankan pipeline fulfillment iSeller Mitra secara manual (sama dengan DAG `etl_iseller_mitra_fulfillment`).
>
> **Flow:**
> 1. Backfill Fulfillment (detect unfulfilled orders -> re-pull API -> update Data Lake)
> 2. Kirim email + Teams report
>
> **Tidak ada re-dump DWH** — pipeline Mitra berhenti di Data Lake.
>
> Token API `access_token_mitra` dan `GMAIL_SMTP_PASSWORD` dibaca dari `.env` (root airflow).

---
## CONFIGURATION

In [ ]:
import datetime
import sys
import os
import pandas as pd

pd.set_option('display.max_columns', None)

# ===================== SET MODE =============================
#    True  = Otomatis (window 9 hari sampai kemarin, seperti DAG)
#    False = Manual (isi START_DATE & END_DATE sendiri)
# ============================================================

AUTO_MODE = False

# ============================================================
#  MODE MANUAL: Isi tanggal di bawah (isi hanya jika AUTO_MODE = False)
# ============================================================

MANUAL_START_DATE = '2026-10-01'
MANUAL_END_DATE   = '2026-10-17'

# ===================== DRY RUN ===============================
#    True  = READ-ONLY. Cuma cek order unfulfilled + tarik API untuk preview.
#            Tidak ada INSERT/UPDATE ke DL, tidak ada email/Teams.
#    False = Run beneran (nulis data + kirim email/Teams)
# ============================================================

DRY_RUN = False

# ============================================================

if AUTO_MODE:
    yesterday = (datetime.date.today() - datetime.timedelta(days=1)).strftime("%Y-%m-%d")
    START_DATE = (datetime.date.today() - datetime.timedelta(days=9)).strftime("%Y-%m-%d")
    END_DATE   = yesterday
else:
    START_DATE = MANUAL_START_DATE
    END_DATE   = MANUAL_END_DATE

print(f"Mode       : {'AUTO (yesterday - 8 days window)' if AUTO_MODE else 'MANUAL'}")
print(f"Period     : {START_DATE} s/d {END_DATE}")
print(f"DRY RUN    : {DRY_RUN}")

---
## LOAD CONFIG & IMPORT PIPELINE

In [ ]:
import importlib, importlib.util

def module_from_file(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

config_dw = module_from_file("config_dw", "../db_config/config_db_datawarehouse.py")

# .env di root airflow: access_token_mitra, GMAIL_SMTP_PASSWORD, POWER_AUTOMATE_TEAMS_WEBHOOK_URL
try:
    from dotenv import load_dotenv
    load_dotenv('../.env')
except ImportError:
    print("python-dotenv tidak terpasang: pastikan access_token_mitra & GMAIL_SMTP_PASSWORD sudah ada di environment.")

# Root airflow (parent folder notebook ini) supaya `common/` di root bisa di-import
AIRFLOW_ROOT = os.path.dirname(os.path.abspath('.'))
if AIRFLOW_ROOT not in sys.path:
    sys.path.insert(0, AIRFLOW_ROOT)

sys.path.insert(0, os.path.join(os.path.abspath('.'), 'pipelines'))

from etl_orchestrator import run_all

print("Config & pipeline loaded successfully.")

---
## RUN PIPELINE

In [ ]:
results = run_all(
    start_date=START_DATE,
    end_date=END_DATE,
    config_dw=config_dw,
    dry_run=DRY_RUN
)

---
## DRY RUN RESULT

> Hanya terisi kalau `DRY_RUN = True`. Data di bawah adalah **persis** yang akan ditulis kalau run asli:
> - `df_headers` — header (dari API) order yang jadi fulfilled. Di Data Lake yang di-UPDATE hanya kolom
>   `fulfillment_status = 'fulfilled'` dan `sales_type`.
> - `df_details` — details yang akan di-append ke `transactions_items_iseller_mitra`.
> - `df_details_skipped` — details order yang sudah punya details di Data Lake (tidak di-append ulang).
> - `df_unfulfilled_dl` — snapshot order unfulfilled dari Data Lake (input backfill).

In [ ]:
frames = ((results or {}).get('backfill') or {}).get('frames', {}) if (results or {}).get('_reason') == 'dry_run' else {}
if not frames:
    print("Tidak ada hasil dry run (DRY_RUN = False, atau tidak ada order unfulfilled).")

df_unfulfilled_dl  = frames.get('df_unfulfilled_dl', pd.DataFrame())
df_headers         = frames.get('df_headers', pd.DataFrame())
df_details         = frames.get('df_details', pd.DataFrame())
df_details_skipped = frames.get('df_details_skipped', pd.DataFrame())

print(f"df_unfulfilled_dl  : {len(df_unfulfilled_dl):,} row ({df_unfulfilled_dl['order_id'].nunique() if len(df_unfulfilled_dl) else 0:,} order)")
print(f"df_headers         : {len(df_headers):,} row ({df_headers['order_id'].nunique() if len(df_headers) else 0:,} order fulfilled)")
print(f"df_details         : {len(df_details):,} row (akan di-append)")
print(f"df_details_skipped : {len(df_details_skipped):,} row (order sudah punya details di DL)")
if len(df_details):
    print(f"\norder_detail_id dobel di df_details: {int(df_details.duplicated(['order_detail_id', 'product_id', 'transactions_type']).sum())}"
          "  (non-0 wajar hanya untuk combo dengan komponen identik)")

---
## REKONSILIASI (header API vs details)

In [ ]:
if len(df_details) and len(df_headers):
    h = df_headers[df_headers['transactions_type'] == 'sale'].drop_duplicates(['order_id', 'transaction_id'])['total_amount'].astype(float).sum()
    d = df_details[df_details['transactions_type'] == 'sale']['total_amount'].sum()
    print(f"header (sale) : {h:,.0f}")
    print(f"details (sale): {d:,.0f}")
    print(f"details/header: {(d / h) * 100:.2f}%" if h else "header = 0")